<a href="https://colab.research.google.com/github/mpragliola/neural-amp-modeler-booster/blob/main/NAM_Profile_Booster.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎸 NAM Profile Booster v2 - by Marco Pragliola

This notebook applies a linear gain boost to Neural Amp Modeller (NAM) profile files by **scaling the output layer weights directly**.
This is useful especially for NAM host ignoring NAM metadata (like `loudness`, `gsin` or `head_scale`), where profiles can sound very quiet.

**How it works:**
- NAM WaveNet models store weights as a flat array
- The final output layer weights are at the end of this array
- Weights are scaled by the desaired factor

**How to use:**
1. Run all cells in order (click on the "play" icon over each cell)
2. Upload your `.nam` file(s) when prompted
3. Set your desired boost in dB
4. Download the boosted file(s)

## 1. Setup

In [ ]:
import json
import os
from google.colab import files

print("✅ Imports complete")

✅ Imports complete


## 2. Upload your NAM file(s)

Run the cell below and select one or more `.nam` files to upload.

In [ ]:
uploaded = files.upload()

nam_files = [f for f in uploaded.keys() if f.endswith('.nam')]

if not nam_files:
    print("⚠️ No .nam files detected. Please upload .nam files.")
else:
    print(f"\n✅ Uploaded {len(nam_files)} NAM file(s):")
    for f in nam_files:
        print(f"   • {f}")

Saving EVH 5150.nam to EVH 5150.nam

✅ Uploaded 1 NAM file(s):
   • EVH 5150.nam


## 3. Set your boost amount

Adjust the `BOOST_DB` value below. Positive values increase volume, negative values decrease it.

**Common values:**
- `+3 dB` = ~1.4x louder (subtle)
- `+6 dB` = ~2x louder (noticeable)
- `+12 dB` = ~4x louder (significant)
- `+20 dB` = ~10x louder (extreme)

In [ ]:
#@title Boost Settings { display-mode: "form" }
#@markdown ### Enter your desired boost in dB:
BOOST_DB = 6.0  #@param {type:"number"}

linear_gain = 10 ** (BOOST_DB / 20)
print(f"🔊 Boost: {BOOST_DB:+.1f} dB ({linear_gain:.4f}x linear gain)")

🔊 Boost: +6.0 dB (1.9953x linear gain)


## 4. Process NAM files

This cell analyzes the model architecture and scales the output layer weights.

In [ ]:
def calculate_wavenet_output_weights_count(config):
    """
    Calculate how many weights belong to the final output layer
    based on the WaveNet architecture config.

    For WaveNet, the output is typically from the last layer's head.
    Looking at the config structure:
    - Last layer has head_size=1, channels=8
    - Output weights = channels * head_size + bias (if head_bias=True)
    """
    if 'layers' not in config:
        return None

    layers = config['layers']
    if not layers:
        return None

    last_layer = layers[-1]

    # The head output weights
    channels = last_layer.get('channels', 8)
    head_size = last_layer.get('head_size', 1)
    head_bias = last_layer.get('head_bias', False)

    # Output layer: channels -> head_size
    # Weights: channels * head_size
    # Bias: head_size (if head_bias is True)
    output_weights = channels * head_size
    if head_bias:
        output_weights += head_size

    return output_weights, channels, head_size, head_bias


def boost_nam_file(input_path, boost_db):
    """
    Apply linear gain boost to a NAM file by scaling output layer weights.

    Args:
        input_path: Path to the input .nam file
        boost_db: Gain boost in decibels

    Returns:
        Path to the output file, or None if failed
    """
    # Load the NAM file
    try:
        with open(input_path, 'r') as f:
            data = json.load(f)
    except json.JSONDecodeError as e:
        print(f"   ❌ Error: {input_path} is not a valid JSON file")
        return None

    # Convert dB to linear gain
    gain = 10 ** (boost_db / 20)

    # Check architecture
    architecture = data.get('architecture', 'unknown')
    print(f"   Architecture: {architecture}")

    if 'weights' not in data:
        print(f"   ❌ No 'weights' array found in file")
        return None

    weights = data['weights']
    if not isinstance(weights, list):
        print(f"   ❌ 'weights' is not a list")
        return None

    total_weights = len(weights)
    print(f"   Total weights: {total_weights:,}")

    # Try to determine output layer size from config
    config = data.get('config', {})
    result = calculate_wavenet_output_weights_count(config)

    if result:
        output_count, channels, head_size, head_bias = result
        print(f"   Last layer: {channels} channels → {head_size} output")
        print(f"   Head bias: {head_bias}")
        print(f"   Output weights to scale: {output_count}")
    else:
        # Fallback: for WaveNet with standard config, output is usually last 8-9 weights
        # (8 weights + 1 bias for single output)
        output_count = 9  # Safe default for most NAM WaveNet models
        print(f"   ⚠️ Could not parse config, using default output_count={output_count}")

    # Scale the last N weights (output layer)
    print(f"   Scaling last {output_count} weights by {gain:.4f}x...")

    for i in range(total_weights - output_count, total_weights):
        weights[i] = weights[i] * gain

    # Update the weights in data
    data['weights'] = weights

    # Generate output filename
    base, ext = os.path.splitext(input_path)
    sign = "+" if boost_db >= 0 else ""
    output_path = f"{base}_{sign}{boost_db:.0f}dB{ext}"

    # Save the modified file (compact JSON to keep file size reasonable)
    with open(output_path, 'w') as f:
        json.dump(data, f, separators=(',', ':'))

    return output_path


# Process all uploaded NAM files
print(f"\n{'='*60}")
print(f"Processing {len(nam_files)} file(s) with {BOOST_DB:+.1f} dB boost")
print(f"{'='*60}\n")

output_files = []

for nam_file in nam_files:
    print(f"📁 Processing: {nam_file}")
    output_path = boost_nam_file(nam_file, BOOST_DB)
    if output_path:
        output_files.append(output_path)
        print(f"   ✅ Saved: {output_path}\n")
    else:
        print(f"   ❌ Failed to process\n")

print(f"{'='*60}")
print(f"✅ Successfully processed {len(output_files)}/{len(nam_files)} files")
print(f"{'='*60}")


Processing 1 file(s) with +6.0 dB boost

📁 Processing: EVH 5150.nam
   Architecture: WaveNet
   Total weights: 13,802
   Last layer: 8 channels → 1 output
   Head bias: True
   Output weights to scale: 9
   Scaling last 9 weights by 1.9953x...
   ✅ Saved: EVH 5150_+6dB.nam

✅ Successfully processed 1/1 files


## 5. Download boosted files

Run the cell below to download your boosted NAM file(s).

In [ ]:
if output_files:
    print("📥 Downloading boosted files...\n")
    for output_file in output_files:
        print(f"   Downloading: {output_file}")
        files.download(output_file)
    print("\n✅ Download complete!")
else:
    print("⚠️ No files to download. Please run the previous cells first.")

📥 Downloading boosted files...

   Downloading: EVH 5150_+6dB.nam


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download complete!


---

## 🔍 Debug: Verify the changes

Run this cell to compare the original and boosted files and confirm the weights were modified.

In [ ]:
#@title Verify weight changes { display-mode: "form" }

if nam_files and output_files:
    original_file = nam_files[0]
    boosted_file = output_files[0]

    with open(original_file, 'r') as f:
        original = json.load(f)
    with open(boosted_file, 'r') as f:
        boosted = json.load(f)

    orig_weights = original['weights']
    boost_weights = boosted['weights']

    print(f"📊 Comparing: {original_file} vs {boosted_file}\n")
    print(f"Total weights: {len(orig_weights):,}")
    print(f"Expected gain: {linear_gain:.4f}x\n")

    # Show last 15 weights comparison
    print("Last 15 weights (output layer):")
    print(f"{'Index':<8} {'Original':<20} {'Boosted':<20} {'Ratio':<10}")
    print("-" * 60)

    for i in range(-15, 0):
        idx = len(orig_weights) + i
        orig_val = orig_weights[i]
        boost_val = boost_weights[i]
        ratio = boost_val / orig_val if orig_val != 0 else float('inf')
        print(f"{idx:<8} {orig_val:<20.6f} {boost_val:<20.6f} {ratio:<10.4f}")

    # Verify unchanged weights at the beginning
    print(f"\nFirst 5 weights (should be UNCHANGED):")
    print(f"{'Index':<8} {'Original':<20} {'Boosted':<20} {'Match':<10}")
    print("-" * 60)

    for i in range(5):
        orig_val = orig_weights[i]
        boost_val = boost_weights[i]
        match = "✅" if orig_val == boost_val else "❌"
        print(f"{i:<8} {orig_val:<20.6f} {boost_val:<20.6f} {match}")
else:
    print("⚠️ No files to compare. Please run the previous cells first.")

📊 Comparing: EVH 5150.nam vs EVH 5150_+6dB.nam

Total weights: 13,802
Expected gain: 1.9953x

Last 15 weights (output layer):
Index    Original             Boosted              Ratio     
------------------------------------------------------------
13787    -0.254810            -0.254810            1.0000    
13788    0.347330             0.347330             1.0000    
13789    -0.286605            -0.286605            1.0000    
13790    -0.323599            -0.323599            1.0000    
13791    0.294428             0.294428             1.0000    
13792    1.265381             1.265381             1.0000    
13793    0.837138             1.670310             1.9953    
13794    1.160673             2.315847             1.9953    
13795    0.621196             1.239449             1.9953    
13796    -1.007838            -2.010901            1.9953    
13797    -0.955654            -1.906780            1.9953    
13798    0.863749             1.723407             1.9953    
13799  

---

## ⚠️ Still too quiet?

If the boost still isn't working, the output layer structure might be different. Run the cell below to manually specify how many weights to scale from the end.

In [ ]:
#@title Manual boost with custom output layer size { display-mode: "form" }
#@markdown If automatic detection isn't working, specify the number of output weights manually:

MANUAL_OUTPUT_COUNT = 9  #@param {type:"integer"}
MANUAL_BOOST_DB = 12.0  #@param {type:"number"}

if nam_files:
    input_file = nam_files[0]

    with open(input_file, 'r') as f:
        data = json.load(f)

    gain = 10 ** (MANUAL_BOOST_DB / 20)
    weights = data['weights']

    print(f"📁 File: {input_file}")
    print(f"🔊 Boost: {MANUAL_BOOST_DB:+.1f} dB ({gain:.4f}x)")
    print(f"🎯 Scaling last {MANUAL_OUTPUT_COUNT} weights\n")

    # Scale
    for i in range(len(weights) - MANUAL_OUTPUT_COUNT, len(weights)):
        weights[i] = weights[i] * gain

    data['weights'] = weights

    # Save
    base, ext = os.path.splitext(input_file)
    sign = "+" if MANUAL_BOOST_DB >= 0 else ""
    output_path = f"{base}_manual_{sign}{MANUAL_BOOST_DB:.0f}dB{ext}"

    with open(output_path, 'w') as f:
        json.dump(data, f, separators=(',', ':'))

    print(f"✅ Saved: {output_path}")
    files.download(output_path)
else:
    print("⚠️ Upload a NAM file first.")